<h2>Part I - Load liabraries and datasets</h2>

In [ ]:
!touch kaggle.json
!echo '{"username":"vietthang0212","key":"699e59a4fc91557a5cfc56e7e35f89c1"}' > kaggle.json

In [ ]:
!touch .mapbox_token
!echo 'pk.eyJ1IjoidGhhbmd0cmFuMDIxMiIsImEiOiJjbGY5NHZzbWwwc2oxM3lvNGx3YjQ5eHp5In0.iIzqRcBRz4n1o4i9E1aoiA' > .mapbox_token

In [ ]:
!pip install kaggle
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# download dataset from Kaggle to Google Colab
! mkdir && cd ~./brazillian_e-commerce_analysis
! kaggle datasets download -d olistbr/brazilian-ecommerce
! unzip brazilian-ecommerce


mkdir: missing operand
Try 'mkdir --help' for more information.
 91% 39.0M/42.6M [00:00<00:00, 138MB/s] 
100% 42.6M/42.6M [00:00<00:00, 119MB/s]
Archive:  brazilian-ecommerce.zip
  inflating: olist_customers_dataset.csv  
  inflating: olist_geolocation_dataset.csv  
  inflating: olist_order_items_dataset.csv  
  inflating: olist_order_payments_dataset.csv  
  inflating: olist_order_reviews_dataset.csv  
  inflating: olist_orders_dataset.csv  
  inflating: olist_products_dataset.csv  
  inflating: olist_sellers_dataset.csv  
  inflating: product_category_name_translation.csv  


In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
# import plotly.io as pio

In [ ]:
df_customers = pd.read_csv('olist_customers_dataset.csv')
df_geolocation = pd.read_csv('olist_geolocation_dataset.csv')
df_order_items = pd.read_csv('olist_order_items_dataset.csv')
df_order_payments = pd.read_csv('olist_order_payments_dataset.csv')
df_order_reviews = pd.read_csv('olist_order_reviews_dataset.csv')
df_orders = pd.read_csv('olist_orders_dataset.csv')
df_products = pd.read_csv('olist_products_dataset.csv')
df_sellers = pd.read_csv('olist_sellers_dataset.csv')
df_product_category_name_translation = pd.read_csv('product_category_name_translation.csv')

In [ ]:
timestamp_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in timestamp_cols:
    df_orders[col] = pd.to_datetime(df_orders[col])

In [ ]:
df_order_items['total_value'] = df_order_items['price'] + df_order_items['freight_value']

df_order_items_ = df_order_items.groupby(by = ['order_id', 'product_id', 'shipping_limit_date'], group_keys = False)[['order_id', 'product_id', 'price', 'total_value']]\
    .agg(
        product_counts = pd.NamedAgg (column = "product_id", aggfunc = "count"),
        total_price = pd.NamedAgg (column = "price", aggfunc = "sum"),
        total_value = pd.NamedAgg (column = "total_value", aggfunc = "sum")
    )\
    .reset_index()


<h2>Part II - Summary and Report</h2>

<h3>1. Customers Analysis</h3>

<h4>1.1. Customers' Spending By Geolocation</h4>

In [ ]:
data_geo = df_geolocation.sort_values(by = 'geolocation_zip_code_prefix', ascending = True)\
    .groupby(by = ['geolocation_city', 'geolocation_state']).head(1)

data_customers = df_orders.merge(df_order_items, how = 'left', on = 'order_id')\
    .query('order_status == "delivered"')\
    .merge(df_customers, how = 'left', on = 'customer_id')\
    .groupby (by = ['customer_city', 'customer_state'])\
    .aggregate({
        'order_id': 'nunique',
        'total_value': 'sum',
    })\
    .sort_values(by = 'order_id', ascending = False)\
    .rename (columns= {'order_id': 'orders_count'})\
    .merge(data_geo, how = 'left', left_on= ['customer_city', 'customer_state'], right_on= ['geolocation_city', 'geolocation_state'])


data_customers['location'] = data_customers['geolocation_city'].apply(lambda x: str(x).title()) + ', ' + data_customers['geolocation_state']

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Bar (
        x = data_customers['location'].head(10),
        y = data_customers['orders_count'].head(10),
        name = 'Number of orders',
        marker_color = 'rgb(64,224,208)',
        hovertemplate = '%{x}' +  '<br>Number of orders: %{y}',
    )
)

fig.add_trace(
    go.Scatter (
        x = data_customers['location'].head(10),
        y = data_customers['total_value'].head(10),
        name = 'GMV',
        yaxis = 'y2',
        marker_color = 'rgb(255,160,122)',
        hovertemplate = '%{x}' + '<br>GMV: $%{y}',
    )
)

fig.update_layout(
    title = 'Top 10 Selling Geolocation',
    plot_bgcolor = 'white',
    yaxis = dict(
        # title=dict(name = 'Number of orders'),
        side = 'left',
        range = [0, max(data_customers['orders_count'])+1000],
        showgrid = False,
        zeroline = True,
        showline = False,
        showticklabels = False,
    ),
    yaxis2 = dict(
        # title = dict(name = 'GMV ($)'),
        side = 'right',
        overlaying = "y",
        showgrid = False,
        zeroline = False,
        showline = False,
        showticklabels = False,
    ),
    xaxis = dict(
        showline = True,
        showgrid = False,
        linecolor = 'rgb(204, 204, 204)',
        linewidth = 1.5,
    ),
    legend = dict(
        orientation="h",
    )
)


In [ ]:
def scaling (x, data, col):
    _min_ = min(data[col])
    _max_ = max(data[col])
    return  (x - _min_)/ (_max_  - _min_)

data_customers['orders_count_scaling'] = data_customers['orders_count'].apply (lambda x: scaling(x, data_customers, 'orders_count'))
data_customers['total_value_scaling'] = data_customers['total_value'].apply (lambda x: scaling(x, data_customers, 'total_value'))

In [ ]:
lat_list = data_customers['geolocation_lat']
lon_list = data_customers['geolocation_lng']

lat_center =  np.mean (lat_list)
lon_center =  np.mean (lon_list)

In [ ]:
data_customers.loc [0:1, 'total_value_scaling'] = data_customers.loc [0:1, 'total_value_scaling']/2

token = open(".mapbox_token").read().strip()

fig = go.Figure(go.Scattermapbox(
        lat = lat_list,
        lon = lon_list,
        mode = 'markers',
        marker = go.scattermapbox.Marker (
            size = data_customers['total_value_scaling']*300,
            sizemin = min(data_customers['total_value']/12),
            color = data_customers['orders_count_scaling']*100,
            cmin = min(data_customers['orders_count'])*100,
            colorscale = 'teal'
        ),
        textposition='top right',
        hoverinfo= 'text',
        hovertext = (
            data_customers['location'].astype(str) +  '<br>' +
            'Number of orders: ' + data_customers['orders_count'].astype(str) + '<br>' +
            'GMV: $' + round(data_customers['total_value'], 2).astype(str)
        ),
    )
)

fig.update_layout(
    autosize = True,
    margin = {"r":5,"t":5,"l":5,"b":5},
    hovermode = 'closest',
    showlegend = False,
    title = dict(
        text = "Consumers' spending by region",
        font = dict(color = 'white'),
        x = 0.05,
        y = 0.9
    ),
    mapbox = dict(
        accesstoken = token,
        bearing = 0,
        center = go.layout.mapbox.Center(
            lat = lat_center,
            lon = lon_center
        ),
        pitch = 0,
        zoom = 3,
        style = "dark"
    )
)

fig.show()


In [ ]:
import json
from urllib.request import urlopen
brazil_geojson_url = 'https://raw.githubusercontent.com/codeforgermany/click_that_hood/main/public/data/brazil-states.geojson'

with urlopen(brazil_geojson_url) as response:
    brazil_states = json.load(response)

data_customers_states = data_customers.groupby (by = 'geolocation_state')[['orders_count', 'total_value']].sum().reset_index()

state_id_map = {}
for feature in brazil_states["features"]:
    feature["id"] = feature["properties"]["sigla"]
    state_id_map[feature["id"]] =  feature["properties"]["id"]

data_customers_states['id'] = data_customers_states["geolocation_state"].apply(lambda x: state_id_map[x]).astype(str)
data_customers_states.head()

,geolocation_state,orders_count,total_value,id
0,AC,80,19575.33,1
1,AL,397,94172.49,2
2,AM,145,27585.47,3
3,AP,67,16141.81,4
4,BA,3242,587373.55,5


In [ ]:

fig = px.choropleth(
    data_customers_states,
    geojson = brazil_states,
    locations = "geolocation_state",
    featureidkey = "properties.sigla",
    color = 'total_value',
    color_continuous_scale = "Oranges",
    labels = {
        'total_value':'GMV',
        "geolocation_state" : 'States'
    },
)

fig.update_layout(
     title = dict (
        text = "Consumers' spending by state",
        x = 0.05,
        y = 0.9
    ),
    margin={"r":5,"t":5,"l":5,"b":5}
)

fig.update_geos (fitbounds = 'locations', visible = False)
fig.show()

In [ ]:
data_customers_states['total_value_scaling'] = data_customers_states['total_value'].apply(lambda x: scaling(x, data_customers_states, 'total_value'))

fig = go.Figure (
    go.Choroplethmapbox (
        geojson = brazil_states,
        locations = data_customers_states['geolocation_state'],
        z = data_customers_states['total_value_scaling'],
        colorscale = "Oranges",
        zmin = 0,
        zmax = 1,
        # marker_opacity = 0.5,
        showscale = False,
        marker_line_width = 0.3,
        text = data_customers['geolocation_state'],
        hoverinfo= 'text',
        hovertext = (
            'States: ' + data_customers_states['geolocation_state'].astype(str) +  '<br>' +
            'GMV: $' + round(data_customers_states['total_value'], 2).astype(str)
        ),
    )
)
fig.update_layout(
    title = dict (
        text = "Consumers' spending by state",
        x = 0.05,
        y = 0.9
    ),
    mapbox = dict(
        accesstoken = token,
        bearing = 0,
        center = go.layout.mapbox.Center(
            lat = lat_center,
            lon = lon_center
        ),
        pitch = 0,
        zoom = 3,
        style = "light"
    ),
    margin={"r":5,"t":5,"l":5,"b":5},
    geo = dict (
        scope = 'south america',

    )
)

fig.update_geos (fitbounds = 'locations', visible = False)

fig.show()

<h4>1.2. Customers Returned and New Customers</h4>

In [ ]:
data_timeseries = df_orders.merge(df_order_items_, how = 'left', on = 'order_id')\
    .query('order_status == "delivered"')\
    .merge(df_customers, how = 'left', on = 'customer_id')\
    .assign (year_month = lambda x: x['order_purchase_timestamp'].dt.to_period('M'))\
    [['order_id', 'customer_unique_id', 'order_purchase_timestamp', 'year_month', 'product_counts', 'total_price', 'total_value']]

data_timeseries['order_index'] = data_timeseries.groupby (by = 'customer_unique_id')['order_purchase_timestamp'].rank (method = 'first', ascending= True)

data_customers_timeseries = data_timeseries.query ('order_index == 1')\
    .groupby (by = 'year_month')\
    .agg (
        new_customers_count = pd.NamedAgg (column = "customer_unique_id", aggfunc = 'nunique'),
        new_customers_value = pd.NamedAgg (column = "total_price", aggfunc = 'sum'),
    )\
    .merge (
        right = (
            data_timeseries.query ('order_index != 1')\
            .groupby (by = 'year_month')\
            .agg (
                return_customers_count = pd.NamedAgg (column = "customer_unique_id", aggfunc = 'nunique'),
                return_customers_value = pd.NamedAgg (column = "total_price", aggfunc = 'sum'),
            )
        ),
        how = 'left',
        on = 'year_month'
    )\
    .fillna (0)\
    .reset_index()

In [ ]:
fig = go.Figure(data = [
    go.Bar (
        x = data_customers_timeseries['year_month'].astype(str),
        y = data_customers_timeseries['new_customers_count'],
        name = 'Number of new customers',
        marker_color = 'mediumaquamarine',
    ),
    go.Bar (
        x = data_customers_timeseries['year_month'].astype(str),
        y = data_customers_timeseries['return_customers_count'],
        name = 'Number of returned customers',
        marker_color = 'powderblue',
    ),
    go.Scatter (
        x = data_customers_timeseries['year_month'].astype(str),
        y = data_customers_timeseries['new_customers_value'],
        name = 'Revenue from new customers',
        yaxis = 'y2',
        marker_color = 'indianred',
        mode = 'lines+markers'
    ),
    go.Scatter (
        x = data_customers_timeseries['year_month'].astype(str),
        y = data_customers_timeseries['return_customers_value'],
        name = 'Revenue from returned customers',
        yaxis = 'y2',
        marker_color = 'sandybrown',
        mode = 'lines+markers'
    )
])

fig.update_layout (
    title = 'New Customers And Return Customers By Month',
    plot_bgcolor = 'white',
    barmode = 'stack',
    xaxis = dict (
        showline = True,
        showgrid = False,
        linecolor = 'rgb(204, 204, 204)',
        linewidth = 1.5,
    ),
    yaxis = dict (
        side = 'left',
        showgrid = False,
        zeroline = True,
        showline = False,
        showticklabels = False,
    ),
    yaxis2 = dict (
        side = 'right',
        overlaying = "y",
        showgrid = False,
        zeroline = False,
        showline = False,
        showticklabels = False,
    ),
    legend = dict (
        orientation="h",
    ),
    hovermode = 'x unified'
)


##### <font color = '#e3f56e'>
Note:
- In general, the majority of customers to this e-commerce sites are new customers with once-in-a-lifetime purchase. From 2017 onwards, it has done a great job in attracting new customers and the expenditure from this group attributed a large portion.<br>
- Returned customers number, on the other hand, stabalised over the periods, and spending from this group is also negligible.
</font>

<h4>1.3. Customers Lifetime Value</h4>

In [ ]:
data_orders =  data_timeseries\
    .groupby (by = ['customer_unique_id'])\
    .agg(
        last_purchase_date = pd.NamedAgg (column = "order_purchase_timestamp", aggfunc = 'max'),
        order_counts = pd.NamedAgg (column = "order_id", aggfunc = 'nunique'),
        quantity = pd.NamedAgg (column = "product_counts", aggfunc = "sum"),
        total_price = pd.NamedAgg (column = "total_price", aggfunc = "sum"),
        total_value	 = pd.NamedAgg (column = "total_value", aggfunc = "sum"),
    )\
    .sort_values (by = 'order_counts', ascending = False)\
    .reset_index()\

data_orders['AOV'] = data_orders['total_price']/data_orders['order_counts']

purchase_freq = data_orders['order_counts'].sum()/ len(data_orders)

repeat_rate = data_orders[data_orders['order_counts'] > 1].shape[0]/ data_orders.shape[0]
churn_rate = 1 - repeat_rate

data_orders['profit_margin'] = data_orders['total_price']* .1

data_orders['CLTV'] = (data_orders['AOV'] * purchase_freq)/ churn_rate *100

<h4>1.4. Customers Segmentation Using RFM Model</h4>

In [ ]:
data_rfm = data_orders[['customer_unique_id', 'last_purchase_date', 'order_counts', 'total_price']].copy(deep = True)\
    .sort_values(by = 'last_purchase_date', ascending = True)

data_rfm.columns = ['customer_unique_id', 'last_purchase_date', 'freq', 'monetary']

recent_date = data_rfm['last_purchase_date'].max()

data_rfm['recency'] = data_rfm['last_purchase_date'].apply (lambda x: (recent_date - x).days).fillna(0).astype(int)

data_rfm['R_rank'] = data_rfm['recency'].rank (ascending = False, method = 'dense', na_option = 'bottom')
data_rfm['F_rank'] = data_rfm['freq'].rank (ascending = True, method = 'dense', na_option = 'bottom')
data_rfm['M_rank'] = data_rfm['monetary'].rank (ascending = True, method = 'dense', na_option = 'bottom')

# normalizing the rank of the customers
data_rfm['R_rank_norm'] = (data_rfm['R_rank']/data_rfm['R_rank'].max())*100
data_rfm['F_rank_norm'] = (data_rfm['F_rank']/data_rfm['F_rank'].max())*100
data_rfm['M_rank_norm'] = (data_rfm['M_rank']/data_rfm['M_rank'].max())*100

data_rfm.drop (columns = ['R_rank', 'F_rank', 'M_rank'], inplace = True)
data_rfm.fillna (0)

data_rfm['RFM_Score'] = (0.15 * data_rfm['R_rank_norm'] + 0.28 *data_rfm['F_rank_norm'] + 0.57 * data_rfm['M_rank_norm']) *.05
data_rfm = data_rfm.round(2).sort_values(by = 'RFM_Score', ascending= False)
data_rfm.head()

,customer_unique_id,last_purchase_date,freq,monetary,recency,R_rank_norm,F_rank_norm,M_rank_norm,RFM_Score
0,8d50f5eadf50201ccdcedfb9e2ac8455,2018-08-20 19:14:26,15,714.63,8,98.69,100.00,89.89,4.70
1,3e43e6105506432c953e165fb2acf44c,2018-02-27 18:36:39,9,1000.85,182,70.21,88.89,94.34,4.46
3,ca77025e7201e3b30c44b472ff346268,2018-06-01 11:38:29,7,806.61,89,85.43,77.78,91.63,4.34
2,6469f99c1f9dfae7733b25662e7f1782,2018-06-28 00:43:34,7,664.20,62,89.85,77.78,88.73,4.29
18,fe81bb32c243a86b2f86fbf053fe6140,2018-06-21 12:10:25,5,1535.40,69,88.71,55.56,97.47,4.22


In [ ]:
def customer_segment (data):
    if data > 4.5:
        return 'Top Customer'
    elif data > 4.0:
        return 'High Value Customer'
    elif data > 3:
        return 'Medium Value Customer'
    elif data > 1.6:
        return 'Low Value Customer'
    else:
        return 'Lost Customer'

data_rfm['segmentation'] = data_rfm['RFM_Score'].apply (lambda x: customer_segment(x))

In [ ]:
def customer_action (data):
    if data['F_rank_norm']*.05 < 2 and data['R_rank_norm']*.05 < 2:
        return 'Hibernating'
    elif 2 <= data['F_rank_norm']*.05 < 4 and data['R_rank_norm']*.05 < 2:
        return 'At Risk'
    elif data['F_rank_norm']*.05 >= 4 and data['R_rank_norm']*.05 < 2:
        return 'Cannot Lose Them'
    elif data['F_rank_norm']*.05 < 2 and 2 <= data['R_rank_norm']*.05 < 3:
        return 'About To Sleep'
    elif 2 <= data['F_rank_norm']*.05 < 3 and 2 <= data['R_rank_norm']*.05 < 3:
        return 'Need Attention'
    elif data['F_rank_norm']*.05 >= 3 and 2 <= data['R_rank_norm']*.05 <= 4:
        return 'Loyal Customer'
    elif data['F_rank_norm']*.05 >= 3 and data['R_rank_norm']*.05 >= 4:
        return 'Champion'
    elif 2 <= data['F_rank_norm']*.05 < 3 and data['R_rank_norm']*.05 >= 3:
        return 'Potential Loyalists'
    elif data['F_rank_norm']*.05 <= 1 and 3 <= data['R_rank_norm']*.05 <= 4:
        return 'Promising'
    else:
        return 'New Customer'

data_rfm['action'] = data_rfm.apply (lambda data_rfm: customer_action(data_rfm), axis = 1)


In [ ]:
data_rfm_summary = data_rfm.groupby (by = 'action')\
    .agg (
        customer_count = pd.NamedAgg (column = 'action', aggfunc = 'count'),
        total_monetary = pd.NamedAgg (column = 'monetary', aggfunc = 'sum'),
    )\
    .reset_index()
data_rfm_summary.loc[9, 'action'] = 'Cannot Lose Them'
data_rfm_summary.fillna(0, inplace = True)
data_rfm_summary['total_monetary_scaling'] = data_rfm_summary['total_monetary'].apply (lambda x: scaling(x, data_rfm_summary, 'total_monetary'))
data_rfm_summary['total_monetary_percent'] = data_rfm_summary['total_monetary'].apply (lambda x: x/ data_rfm_summary['total_monetary'].sum()*100).astype('float16')


In [ ]:
color = {}
for ele in data_rfm_summary[['action', 'total_monetary_scaling']].to_dict('records'):
    if ele['total_monetary_scaling'] <= 0.1:
        color[ele['action']] = '#ffcab3'
    elif ele['total_monetary_scaling'] <= 0.3:
        color[ele['action']] = '#ffb999'
    elif ele['total_monetary_scaling'] <= 0.5:
        color[ele['action']] = '#ffb999'
    elif ele['total_monetary_scaling'] <= 0.8:
        color[ele['action']] = '#ff9566'
    else:
        color[ele['action']] = '#ff6119'


In [ ]:
def property (x, y, name):
    return go.Scatter (
        x = x,
        y = y,
        fill = 'toself',
        fillcolor = color[name],
        hoveron = 'fills',
        hoverinfo = 'text',
        line_color = 'white',
        mode = 'lines+text',
        name = name,
    )

In [ ]:
def annotation (x, y, name):
    monetary = float(data_rfm_summary[data_rfm_summary['action'] == name]['total_monetary'])
    monetary_percent = round (float (data_rfm_summary[data_rfm_summary['action'] == name]['total_monetary_percent']),2)
    customers_count = int(data_rfm_summary[data_rfm_summary['action'] == name]['customer_count'])

    text = f'<b>{name}</b><br>Total Customers: {customers_count}<br>Total Monetary: {monetary} ({monetary_percent}%)'

    return fig.add_annotation (
        x = x, y = y, xref = 'x domain', yref= 'y domain', font = dict(color = 'black', size = 11),text = text, align= 'left', xanchor = 'left', showarrow = False
    )

In [ ]:
fig = go.Figure()

fig.add_trace(
    property (
        x = [0, 0, 2, 2],
        y = [0, 2, 2, 0],
        name = "Hibernating"
    )
)
fig.add_trace(
    property (
        x = [0, 0, 2, 2],
        y = [2, 4, 4, 2],
        name = "At Risk",
    )
)
fig.add_trace(
    property (
        x = [0, 0, 2, 2],
        y = [4, 5, 5, 4],
        name = "Cannot Lose Them",
    )
)
fig.add_trace(
    property (
        x = [2, 2, 4, 4],
        y = [3, 5, 5, 3],
        name = "About To Sleep",
    )
)
fig.add_trace(
    property (
        x = [2, 2, 3, 3],
        y = [2, 3, 3, 2],
        name = "Need Attention",
    )
)
fig.add_trace(
    property (
        x = [2, 2, 3, 3],
        y = [0, 2, 2, 0],
        name = "Loyal Customer",
    )
)
fig.add_trace(
    property (
        x = [4, 4, 5, 5],
        y = [3, 5, 5, 3],
        name = "Champion",
    )
)
fig.add_trace(
    property (
        x = [3, 3, 5, 5],
        y = [1, 3, 3, 1],
        name = "Potential Loyalists",
    )
)
fig.add_trace(
    property (
        x = [3, 3, 4, 4],
        y = [0, 1, 1, 0],
        name = "Promising",
    )
)
fig.add_trace(
    property (
        x = [4, 4, 5, 5],
        y = [0, 1, 1, 0],
        name = "New Customer",
    )
)

annotation (x = 0.01, y = 0.01, name = 'Hibernating')
annotation (x = 0.01, y = 0.51, name = 'At Risk')
annotation (x = 0.01, y = 0.98, name = 'Cannot Lose Them')
annotation (x = 0.41, y = 0.01, name = 'Loyal Customer')
annotation (x = 0.41, y = 0.50, name = 'Need Attention')
annotation (x = 0.41, y = 0.91, name = 'About To Sleep')
annotation (x = 0.81, y = 0.91, name = 'Champion')
annotation (x = 0.61, y = 0.41, name = 'Potential Loyalists')
annotation (x = 0.61, y = 0.01, name = 'Promising')
annotation (x = 0.81, y = 0.01, name = 'New Customer')

fig.update_layout(
    title = "Customer Segmentation by RFM Score",
    plot_bgcolor = 'white',
    xaxis = dict (
        title = dict (
            text = 'Recency Score'
        ),
        showline = True,
        range = [0, 5],
        tickwidth = 2,
        ticklen = 7,
        rangemode = 'nonnegative'
    ),
    yaxis = dict (
        title = dict (
            text = 'Frequency Score'
        ),
        showline = True,
        range = [0, 5],
        tickwidth = 2,
        ticklen = 7,
        rangemode = 'nonnegative'
    ),
    showlegend = False
)


##### <font color = '#e3f56e'>
Note:
- Most of the customers have a relatively low frequency and low recency of purchasing on the site. This group dosen't bring any sustainable revenue. If we try to get them back, then the cost of asquisition might be higher than the expected profit.<br>
- One other purchasing group is that they purchase frequently but not recently for a while ago. They are loyal users but for some reasons, they are about to leave, so bonus and promotion is needed to attract them.<br>
- The highest proportion is new customers, which takes up for 50% of total revenue. Users have recently made the purchase but not frequently so we need to try to encourage them to buy more.<br>
</font>

<h3>2. Product Analysis</h3>

In [ ]:
data_products = df_order_items_.groupby (by = 'product_id')[['product_counts', 'total_price']].sum()\
    .merge(df_products, how = 'left', on = 'product_id')\
    .groupby(by = ['product_category_name'])[['product_counts', 'total_price']].sum()\
    .sort_values(by = ['product_counts'], ascending= False)\
    .reset_index()

data_products['product_category'] = data_products['product_category_name'].apply (lambda x: str(x).replace('_', ' ').title())

<h4>2.1. Top Products Sold By Category</h4>

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Bar (
        x = data_products['product_category'].head(15),
        y = data_products['product_counts'].head(15),
        name = 'Number of products sold',
        marker_color = 'rgb(64,224,208)',
    )
)

fig.add_trace(
    go.Scatter (
        x = data_products['product_category'].head(15),
        y = data_products['total_price'].head(15),
        name = 'GMV',
        yaxis = 'y2',
        marker_color = 'rgb(255,160,122)',
        mode = 'lines'
        # hovertemplate = 'GMV: $%{y}',
    )
)

fig.update_layout (
    title = 'Top 10 Selling Product Categories',
    plot_bgcolor = 'white',
    xaxis = dict (
        showline = True,
        showgrid = False,
        linecolor = 'rgb(204, 204, 204)',
        linewidth = 1.5,
    ),
    yaxis = dict (
        side = 'left',
        showgrid = False,
        zeroline = True,
        showline = False,
        showticklabels = False,
    ),
    yaxis2 = dict (
        side = 'right',
        overlaying = "y",
        showgrid = False,
        zeroline = False,
        showline = False,
        showticklabels = False,
    ),
    legend = dict (
        orientation="h",
    ),
    hovermode = 'x'
)


<h4>2.2. Products Segmentation</h4>

In [ ]:
data_products_seg = df_order_items.merge (df_orders[['order_id', 'order_status']].query('order_status == "delivered"'), how = 'inner', on = 'order_id')\
    .groupby (by = 'product_id')\
    .agg(
        sales_volume = pd.NamedAgg (column = 'product_id', aggfunc = 'count'),
        sales_values = pd.NamedAgg (column = 'price', aggfunc = 'sum'),
    )\
    .sort_values (by = 'sales_values', ascending = False)\
    .reset_index()

In [ ]:
data_sales_his = df_orders.query ('order_status == "delivered"')[['order_id', 'order_purchase_timestamp']]\
    .merge (df_order_items[['order_id', 'product_id']], how = 'left', on = 'order_id')

data_sales_his['order_purchase_date'] = data_sales_his['order_purchase_timestamp'].dt.date
data_sales_his = data_sales_his[['product_id', 'order_purchase_date']]\
    .groupby (by = ['order_purchase_date', 'product_id'])\
    .agg (
        product_sold = pd.NamedAgg (column = 'product_id', aggfunc = 'count')
    )\
    .reset_index()

data_sales_his['day'] = data_sales_his['order_purchase_date'].rank(method = 'dense', ascending = True).astype (int)
data_sales_his = data_sales_his.pivot(index = 'product_id', columns = ['day'], values = 'product_sold').fillna(0).astype(int).reset_index()
data_sales_his.columns = [data_sales_his.columns[0]] + ['day_' + str(col) for col in data_sales_his.columns[1:]]

sales_col = data_sales_his.columns[1:]
data_sales_his = data_sales_his.merge (data_products_seg, how = 'left', on = 'product_id')
data_sales_his['mean'] = data_sales_his[sales_col].mean(axis = 1)
data_sales_his = data_sales_his.query ('mean > 0')
data_sales_his['std'] = data_sales_his[sales_col].std(axis = 1)
data_sales_his['coef'] = data_sales_his['std']/data_sales_his['mean']

data_sales_his.head()

,product_id,day_1,day_2,day_3,day_4,day_5,day_6,day_7,day_8,day_9,...,day_608,day_609,day_610,day_611,day_612,sales_volume,sales_values,mean,std,coef
0,00066f42aeeb9f3007548bb9d3f33c38,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,101.65,0.001634,0.040423,24.738634
1,00088930e925c41fd95ebfe695fd2655,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,129.90,0.001634,0.040423,24.738634
2,0009406fd7479715e4bef61dd91f2462,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,229.00,0.001634,0.040423,24.738634
3,000b8f95fcb9e0096488278317764d19,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,2,117.80,0.003268,0.057119,17.478535
4,000d9be29b5207b54e86aa1b1ac54872,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,199.00,0.001634,0.040423,24.738634


In [ ]:
data_product_summary = data_sales_his.copy().drop (sales_col, axis = 1)
total_sales_values = data_product_summary['sales_values'].sum()
data_product_summary['%sales'] = data_product_summary['sales_values']/total_sales_values*100
data_product_summary.sort_values (by = '%sales', ascending = False, inplace = True, ignore_index = True)
data_product_summary['%sales_CS'] = data_product_summary['%sales'].cumsum()


In [ ]:
n_products = len(data_product_summary)
n_a, n_b = int(0.05 * n_products), int(0.2 * n_products)

data_product_summary['sku_id'] = pd.Series (range(1, n_products + 1)).astype(int)
data_product_summary['abc'] = pd.Series (range (n_products)).apply (lambda x: 'A' if x <= n_a -1 else 'B' if x <= n_b - 1 else 'C')
data_product_summary

,product_id,sales_volume,sales_values,mean,std,coef,%sales,%sales_CS,sku_id,abc
0,bb50f2e236e5eea0100680137654686c,194,63560.00,0.316993,0.728767,2.298998,0.480732,0.480732,1,A
1,6cdd53843498f92890544667809f1595,153,53652.30,0.250000,0.614540,2.458161,0.405796,0.886528,2,A
2,d6160fb7873f184099d9bc95e30376af,33,45949.35,0.053922,0.331689,6.151316,0.347535,1.234063,3,A
3,d1c427060a0f73f6b889a5c7c61f2ac4,332,45620.56,0.542484,0.987562,1.820446,0.345048,1.579112,4,A
4,99a4788cb24856965c36a24e339b6058,477,42049.66,0.779412,1.211795,1.554755,0.318040,1.897152,5,A
...,...,...,...,...,...,...,...,...,...,...
32211,2e8316b31db34314f393806fd7b6e185,1,2.99,0.001634,0.040423,24.738634,0.000023,99.999925,32212,C
32212,680cc8535be7cc69544238c1d6a83fe8,1,2.90,0.001634,0.040423,24.738634,0.000022,99.999947,32213,C
32213,8a3254bee785a526d548a81a9bc3c9be,3,2.55,0.004902,0.090328,18.427014,0.000019,99.999966,32214,C
32214,310dc32058903b6416c71faff132df9e,1,2.29,0.001634,0.040423,24.738634,0.000017,99.999983,32215,C


In [ ]:
for product_class in ['A', 'B', 'C']:
    filter = data_product_summary[data_product_summary['abc'] == product_class]
    percentage = len(filter)/ len(data_product_summary)*100
    sku_number = filter['sku_id'].max()
    sales_percent = filter['%sales'].sum()

    print(f'Class {product_class} ({percentage:.0f}%) has {sku_number} SKUs and {sales_percent:.2f}% of turnover')

Class A (5%) has 1610 SKUs and 47.82% of turnover
Class B (15%) has 6443 SKUs and 27.00% of turnover
Class C (80%) has 32216 SKUs and 25.18% of turnover


In [ ]:
import math
data_product_plot = data_product_summary.copy()
class_mapping = {'A': 'mediumturquoise', 'B': 'indianred', 'C': 'limegreen'}
fig = go.Figure()

for prod_class in class_mapping.keys():
    fig.add_trace(
        go.Scatter (
            x = data_product_plot[data_product_plot['abc'] == prod_class]['%sales'],
            y = data_product_plot[data_product_plot['abc'] == prod_class]['coef'],
            name = 'Product class ' + str(prod_class),
            marker = dict (
                color = class_mapping[prod_class]
            ),
            mode = 'markers'
        ),
    )

fig.update_layout (
    title = 'Distribution by Demand Variablity',
    plot_bgcolor = 'white',
    xaxis = dict (
        title = 'Percentage of Turnover (%)',
        showline = True,
        showgrid = False,
        linecolor = 'rgb(204, 204, 204)',
        linewidth = 1.5,
        range = [math.floor(min(data_product_summary['%sales'])) - 0.01, (max(data_product_summary['%sales'])) + 0.01]
    ),
    yaxis = dict (
        title = 'Variability of the demand',
        showline = True,
        showgrid = False,
        linecolor = 'rgb(204, 204, 204)',
        linewidth = 1.5,
        range = [math.floor(min(data_product_summary['coef'])), math.ceil(max(data_product_summary['coef'])) + 1]
    ),
    legend = dict (
        orientation="h",
    )
)

In [ ]:
from plotly.subplots import make_subplots

fig = make_subplots (rows = 1, cols = 3, start_cell = 'top-left', subplot_titles = ('Class A', 'Class B', 'Class C'))
product_classes = ['A', 'B', 'C']
col = 1
for product_class in product_classes:
    x = data_product_summary[data_product_summary['abc'] == product_class]['sales_values']
    # y = data_product_summary[data_product_summary['abc'] == product_class]['mean']

    trace = go.Histogram(x = x, nbinsx = 50)
    fig.append_trace (trace, 1, col)
    fig.update_xaxes (title_text = "Sales Values", row = 1, col = col)
    fig.update_yaxes (title_text = "Product Count", row = 1, col = col)
    fig.update_layout (
        title = 'Sales Distribution By Product Class',
        plot_bgcolor = 'white',
        showlegend = False,
        bargap = 0.03
    )
    col += 1

fig.show()

In [ ]:
volatility_threshold = (max(data_product_summary['std']) - min(data_product_summary['std']))/ 2
def classify (product):
    if product['%sales'] <= 0.25 and product['std'] <= volatility_threshold:
        return 'Low Volume, Low Volatility'
    elif product['%sales'] > 0.25 and product['std'] <= volatility_threshold:
        return 'High Volume, Low Volatility'
    elif product['%sales'] <= 0.25 and product['std'] > volatility_threshold:
        return 'Low Volume, High Volatility'
    else:
        return 'High Volume, High Volatility'

data_product_summary['type'] = data_product_summary.apply(lambda x: classify(x), axis = 1)

In [ ]:
type_mapping = {
    'Low Volume, Low Volatility': {
        'info' : 'Easy/ Low ROI',
        'color': 'powderblue',
        'x': [0, 2, 2, 0],
        'y': [0, 0, 2, 2]
    },
    'Low Volume, High Volatility': {
        'info' : 'Difficult/ Low ROI',
        'color': 'orangered',
        'x': [0, 0, 2, 2],
        'y': [2, 4, 4, 2],
    },
    'High Volume, Low Volatility': {
        'info' : 'Moderate+/ High ROI',
        'color': 'skyblue',
        'x': [2, 4, 4, 2],
        'y': [0, 0, 2, 2],
    },
    'High Volume, High Volatility': {
        'info' : 'Critical+/ Moderate+/ High ROI',
        'color': 'yellowgreen',
        'x': [2, 4, 4, 2],
        'y': [2, 2, 4, 4],
    }
}
def plot (x, y, _type_):
    return go.Scatter (
        x = x,
        y = y,
        fill = 'toself',
        fillcolor = type_mapping[_type_]['color'],
        hoveron = 'fills',
        hoverinfo = 'text',
        line_color = 'white',
        mode = 'lines+text',
        name = _type_,
    )
total_values = data_product_summary['sales_values'].sum()
def plot_annotation (x, y, _type_):
    total_products_sold = data_product_summary[data_product_summary['type'] == _type_]['product_id'].nunique()
    total_sales_volume = data_product_summary[data_product_summary['type'] == _type_]['sales_volume'].sum()
    total_sales_values = data_product_summary[data_product_summary['type'] == _type_]['sales_values'].sum()
    values_percent = total_sales_values/total_values * 100
    info = type_mapping[_type_]['info']
    text = f'<b>{_type_}</b><br>{info}<br>Total Products Sold: {total_products_sold}<br>Total Sales Volume: {total_sales_volume}<br>Total Sales Value: {total_sales_values:.2f} ({values_percent:.2f}%)'
    return fig.add_annotation (
        x = x, y = y , font = dict(color = 'black', size = 12),text = text, align= 'left', xanchor = 'left', showarrow = False
    )

In [ ]:
fig = go.Figure()
for _type_ in set(data_product_summary['type']):
    x = type_mapping[_type_]['x']
    y = type_mapping[_type_]['y']
    fig.add_trace (
        plot (x, y, _type_)
    )
    plot_annotation (x[0] + 0.05, y[0] + 0.7, _type_)

fig.update_layout(
    title = "Product Segmentation by Volume and Volatility",
    plot_bgcolor = 'white',
    xaxis = dict (
        title = dict (
            text = 'Volatility'
        ),
        showline = False,
        range = [0, 4],
        rangemode = 'nonnegative'
    ),
    yaxis = dict (
        title = dict (
            text = 'Volume'
        ),
        showline = False,
        range = [0, 4],
        rangemode = 'nonnegative'
    ),
    showlegend = False
)

<h2>Part III: E-commerce Operation Analysis</h2>

<h3>3. Time-series Analysis</h3>

<h4>3.1. Daily GMV</h4>

In [ ]:
data_orders_timeseries_daily = df_orders.merge(df_order_items_, how = 'left', on = 'order_id')\
    .query('order_status == "delivered"')

data_orders_timeseries_daily['date'] = data_orders_timeseries_daily['order_delivered_customer_date'].dt.to_period('D')

data_orders_timeseries_daily = data_orders_timeseries_daily.groupby ('date')\
    .agg({
        'order_id': 'count',
        'product_counts': 'sum',
        'total_price': 'sum',
    })\
    .query ("date.isna() == False")\
    .reset_index()

data_orders_timeseries_daily['7d_moving_average'] = data_orders_timeseries_daily.sort_values(by = 'date') [['total_price']]\
    .transform(lambda x: round (x.rolling(7).mean(), 2))

data_orders_timeseries_daily['30d_moving_average'] = data_orders_timeseries_daily.sort_values(by = 'date') [['total_price']]\
    .transform(lambda x: round (x.rolling(30).mean(), 2))

data_orders_timeseries_daily['index'] = data_orders_timeseries_daily.index + 1

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter (
        x = data_orders_timeseries_daily['date'].astype(str),
        y = data_orders_timeseries_daily['total_price'],
        mode = 'markers',
        name = 'GMV',
        marker_color = '#ffa07a',
    )
)

fig.add_trace(
    go.Scatter (
        x = data_orders_timeseries_daily['date'].astype(str),
        y = data_orders_timeseries_daily['7d_moving_average'],
        mode = 'lines',
        name = '7-D Moving Average',
        marker_color = '#03b6fc',
    )
)

fig.add_trace(
    go.Scatter (
        x = data_orders_timeseries_daily['date'].astype(str),
        y = data_orders_timeseries_daily['30d_moving_average'],
        mode = 'lines',
        name = '30-D Moving Average',
        marker_color = '#1de02d',
    )
)

fig.update_layout (
    title = 'Daily revenue',
    plot_bgcolor = 'white',
    xaxis = dict (
        showline = True,
        showgrid = False,
        linecolor = 'rgb(204, 204, 204)',
        linewidth = 1.5,
    ),
    yaxis = dict (
        showgrid = False,
        zeroline = False,
        showline = False,
        showticklabels = True,
        linecolor = 'rgb(204, 204, 204)',
        linewidth = 1.5,
    ),
    legend = dict(
        orientation = 'h'
    ),
    hovermode = 'x unified'
)


<font color = '#e3f56e'>
Note: <br>
- The daily GMV value flactuated over the period, which can range from 0 to over 60.000. However, the overall trend was climbing up. <br>
- One of abnormal point here is from Oct 2016 to Jan 2017, and from Oct 2018, the GMV seems to flatten down before 5.000.
</font>

<h4>3.2 Monthly GMV</h4>

In [ ]:
data_orders_timeseries = df_orders.merge(df_order_items_, how = 'left', on = 'order_id')\
    .query('order_status == "delivered"')

data_orders_timeseries['year_month'] = data_orders_timeseries['order_delivered_customer_date'].dt.to_period('M')

data_orders_timeseries = data_orders_timeseries.groupby ('year_month')\
    .agg({
        'order_id': 'count',
        'product_counts': 'sum',
        'total_price': 'sum',
    })\
    .query ("year_month.isna() == False")\
    .rename (columns={'order_id': 'orders_count'})\
    .reset_index()

data_orders_timeseries['moving_average'] = data_orders_timeseries.sort_values(by = 'year_month') [['total_price']]\
    .transform(lambda x: round (x.rolling(3).mean(), 2))\
    .fillna(0)


In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Bar (
        x = data_orders_timeseries['year_month'].astype(str),
        y = data_orders_timeseries['orders_count'],
        name = 'Number of orders',
        marker_color = 'rgb(64,224,208)',
    )
)

fig.add_trace(
    go.Scatter (
        x = data_orders_timeseries['year_month'].astype(str),
        y = data_orders_timeseries['total_price'],
        name = 'GMV',
        yaxis = 'y2',
        marker_color = 'rgb(255,160,122)',
        mode = 'lines'
    )
)

fig.update_layout (
    title = 'Monthly GMV',
    plot_bgcolor = 'white',
    xaxis = dict (
        showline = True,
        showgrid = False,
        linecolor = 'rgb(204, 204, 204)',
        linewidth = 1.5,
    ),
    yaxis = dict (
        side = 'left',
        showgrid = False,
        zeroline = True,
        showline = False,
        showticklabels = False,
    ),
    yaxis2 = dict (
        side = 'right',
        overlaying = "y",
        showgrid = False,
        zeroline = False,
        showline = False,
        showticklabels = False,
    ),
    legend = dict (
        orientation="h",
    ),
    hovermode = 'x'
)


<h4>3.3. Sales Prediction Model</h4>

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

data_orders_timeseries['total_price_diff'] = data_orders_timeseries['total_price'].diff()
data_orders_timeseries['month'] = data_orders_timeseries.index + 1

sales_data = data_orders_timeseries[['total_price_diff']].copy (deep = True).fillna(0)
sales_actual = data_orders_timeseries['total_price'].to_list()

for month in range (1, 13):
    col_name = 'month_' + str(month)
    sales_data[col_name] = sales_data['total_price_diff'].shift(month)

sales_data = sales_data.dropna ().reset_index(drop = True)

train_data = sales_data[:10]
test_data = sales_data[10:]

scaler = MinMaxScaler (feature_range = (-1,1))
scaler.fit (train_data)

train_data = scaler.transform (train_data)
test_data = scaler.transform (test_data)
X_train, y_train = train_data[:,1:], train_data[:,0:1].ravel()
X_test, y_test = test_data[:,1:], test_data[:,0:1].ravel()

model = make_pipeline(
        SplineTransformer(
            knots = np.linspace(0, np.pi**2 + 5, 36).reshape (3, 12),
            degree = 2,
            extrapolation = "periodic"
        ),
        Ridge (alpha = 1e-8)
    )
model.fit (X_train, y_train)

y_pred = model.predict(X_test).reshape (-1, 1)
y_pred = np.concatenate([y_pred, X_test], axis = 1)
sales_pred = scaler.inverse_transform (y_pred)
result = []
for index in range (0, len(sales_pred)):
    if (sales_pred[index][0] + sales_actual[index]) < 0:
        result.append (0)
    else:
        result.append (sales_pred[index][0] + sales_actual[index])

accuracy = r2_score (sales_actual[-3:], result)
print (f'The accuracy is {accuracy*100:.2f}%')

The accuracy is 72.91%


In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter (
        x = data_orders_timeseries['year_month'].astype(str),
        y = data_orders_timeseries['total_price'],
        name = 'Actual Sales',
        marker_color = 'rgb(255,160,122)',
        mode = 'lines+markers'
    )
)

fig.add_trace(
    go.Scatter (
        x = data_orders_timeseries['year_month'].astype(str).to_list()[-3:],
        y = result,
        name = 'Predicted Sales',
        marker_color = 'lightseagreen',
        mode = 'lines+markers'
    )
)

fig.update_layout (
    title = 'Monthly GMV',
    plot_bgcolor = 'white',
    xaxis = dict (
        showline = True,
        showgrid = False,
        linecolor = 'rgb(204, 204, 204)',
        linewidth = 1.5,
    )
)

<h4>3.4. Customers' Satisfaction</h4>

In [ ]:
data_review = df_order_reviews[['order_id', 'review_score']]\
    .groupby(by = 'order_id')\
    .agg({'review_score':'mean'})\
    .reset_index()

In [ ]:
data_orders_ = df_orders.copy (deep = True)\
    .merge (data_review, how = 'left', on = 'order_id')\
    .merge (df_order_items[['order_id', 'seller_id', 'freight_value']], how = 'left', on = 'order_id')\
    .merge (df_sellers[['seller_id', 'seller_city', 'seller_state']], how = 'left', on = 'seller_id')\
    .merge (df_customers[['customer_id', 'customer_city', 'customer_state']], how = 'left', on = 'customer_id')\
    .drop_duplicates (keep = 'first')\
    .reset_index (drop = True)

data_orders_['review_score'].fillna(0, inplace = True)

In [ ]:
data_orders_['year_month'] = data_orders_['order_purchase_timestamp'].dt.to_period ('M')
data_orders_['approved_mins'] = ((data_orders_['order_approved_at'] - data_orders_['order_purchase_timestamp']).dt.total_seconds()/60).fillna(0).astype(int)
data_orders_['to_carrier_days'] = (data_orders_['order_delivered_carrier_date'] - data_orders_['order_approved_at']).dt.days.fillna(0).astype(int)
data_orders_['to_customer_days'] = (data_orders_['order_delivered_customer_date'] - data_orders_['order_delivered_carrier_date']).dt.days.fillna(0).astype(int)
data_orders_['delivery_days'] = data_orders_['to_carrier_days'] + data_orders_['to_customer_days']
data_orders_['is_late'] = np.where (data_orders_['order_estimated_delivery_date'] <= data_orders_['order_delivered_customer_date'], True, False)

In [ ]:
columns_list = ['order_id', 'customer_id', 'customer_city', 'customer_state', 'seller_id', 'seller_city',
       'seller_state', 'order_status', 'order_purchase_timestamp',
       'review_score',  'freight_value',  'year_month',
       'approved_mins', 'to_carrier_days', 'to_customer_days', 'delivery_days', 'is_late']

data_orders_ = data_orders_[columns_list]


In [ ]:
fig = go.Figure(
    go.Histogram(
        x = data_orders_['review_score'],
        histfunc = 'count',
        texttemplate = "%{y}",
        name = 'Review score',
        textfont_size = 12,
        textposition = 'outside',
        hovertemplate = 'Review score: %{x}<br>No. of orders: %{y}',
        xbins = dict (
            start = -1.0,
            end = 5.0,
            size = 1.0
        )
    )
)

fig.update_layout(
    title = "Customer review",
    plot_bgcolor = 'white',
    xaxis = dict (
        title = 'Review score',
    ),
    yaxis = dict (
        title = 'Order numbers',
        showticklabels = False
    )
)
fig.show()

<h4>3.5. Shipment Complettion Ratio</h4>

In [ ]:
data_orders_status = df_orders.groupby(by = 'order_status', group_keys= False)['order_id'].count().reset_index()

fig = px.pie(
    data_orders_status,
    values='order_id',
    names='order_status',
    title='Order status numbers',
    # color_discrete_sequence=px.colors.sequential.Blues_r,
    hover_data=['order_status'], labels={'order_id':'numbers of orders', 'order_status': 'order status'}
)

fig.update_traces(textposition='inside', textinfo = 'percent+label')
fig.show()

In [ ]:
total_shipments = data_orders_['order_id'].nunique()
total_cancelled_shipments = data_orders_[data_orders_['order_status'] == 'canceled']['order_id'].nunique()

print (f'Cancelled shipment ratio is {(total_cancelled_shipments/ total_shipments)*100:.2f}%')

Cancelled shipment ratio is 0.63%


In [ ]:
total_derlivered_shipments = data_orders_[data_orders_['order_status'] == 'delivered']['order_id'].nunique()
total_late_delivered_shipments = data_orders_[(data_orders_['order_status'] == 'delivered') & (data_orders_['is_late'] == False)]['order_id'].nunique()

print (f'On-time delivered shipment ratio is {(total_late_delivered_shipments/ total_derlivered_shipments)*100:.2f}%')

On-time delivered shipment ratio is 91.89%


In [ ]:
avg_confirmed_mins = data_orders_.query ('approved_mins != 0').groupby(by = 'order_id').agg({'approved_mins':'mean'})['approved_mins'].mean()
avg_delivery_days = data_orders_.query ('delivery_days != 0').groupby(by = 'order_id').agg({'delivery_days':'mean'})['delivery_days'].mean()

print (f'For each order, it takes approximately {avg_confirmed_mins:.0f} minute(s) on average for an order to be approved.')
print (f'For each order, it takes approximately {avg_delivery_days:.0f} day(s) on average to deliver to customer after approved.')

For each order, it takes approximately 633 minute(s) on average for an order to be approved.
For each order, it takes approximately 11 day(s) on average to deliver to customer after approved.


In [ ]:
data_orders_year_month = data_orders_[['order_id', 'year_month', 'order_status']]\
    .drop_duplicates()\
    .groupby (by = ['year_month', 'order_status'])\
    .agg (
        orders_count = pd.NamedAgg (column = 'order_status', aggfunc = 'count')
    )\
    .reset_index()\
    .pivot (index = 'year_month', columns = 'order_status', values = 'orders_count')\
    .reset_index()\
    .fillna(0)\
    # .drop(columns = 'order_status')

In [ ]:
year_month_list = data_orders_year_month['year_month'].astype(str).to_list()
order_status_list = list(set(data_orders_['order_status']))
fig = go.Figure()

for order_status in order_status_list:
    fig.add_trace(go.Scatter(
        x = year_month_list,
        y = data_orders_year_month[order_status].to_list(),
        name = order_status,
        mode = 'lines',
        line = dict (width = 0.5),
        stackgroup = 'one' # define stack group
    ))

fig.update_layout(
    title = "Orders by status",
    plot_bgcolor = 'white',
    hovermode = 'x unified',
    # showlegend = False
)

fig.show()

In [ ]:
data_orders_year_month['total_orders'] = 0

for status in set(data_orders_['order_status']):
    data_orders_year_month['total_orders'] += data_orders_year_month[status]
avg_orders_by_month = data_orders_year_month['total_orders'].mean()
print (f'On average, {avg_orders_by_month:.0f} orders created are per month.')

total_orders = data_orders_year_month['total_orders'].sum()
total_days = (data_orders_['order_purchase_timestamp'].max() - data_orders_['order_purchase_timestamp'].min()).days
print (f'On average, {total_orders/total_days:.0f} orders are created per day.')

total_hours = total_days*24
print (f'On average, {total_orders/total_hours:.0f} orders are created per hour.')

On average, 3978 orders created are per month.
On average, 129 orders are created per day.
On average, 5 orders are created per hour.


In [ ]:
temp = data_orders_[['order_id', 'order_purchase_timestamp']]\
    .copy (deep = True)\
    .sort_values (by = 'order_purchase_timestamp')\
    .drop_duplicates()

temp['order_purchase_diff'] = temp['order_purchase_timestamp'].diff().astype ('timedelta64[m]').fillna(0).astype(int)
avg_diff = temp['order_purchase_diff'].mean()

print (f'On average, after {avg_diff:.0f} minutes an order would be created.')

On average, after 11 minutes an order would be created.


<h4>3.6. Evaluation by sellers</h4>

In [ ]:
data_shipping = data_orders_[['order_id', 'seller_city', 'order_status', 'seller_state', 'customer_city', 'customer_state', 'freight_value', 'to_carrier_days', 'to_customer_days','delivery_days']]\
    .query ('order_status == "delivered"')\
    .groupby (by = ['seller_city', 'seller_state', 'customer_city', 'customer_state'])\
    .agg (
        orders_num_same_route = pd.NamedAgg (column = 'order_id', aggfunc = 'nunique'),
        avg_freight_value = pd.NamedAgg (column = 'freight_value', aggfunc = 'mean'),
        avg_delivery_days = pd.NamedAgg (column = 'delivery_days', aggfunc = 'mean'),
    )\
    .reset_index()
obj_type = {
    'orders_num_same_route': 'int',
    'avg_freight_value': 'float',
    'avg_delivery_days': 'int',
}
data_shipping.astype (obj_type)

,seller_city,seller_state,customer_city,customer_state,orders_num_same_route,avg_freight_value,avg_delivery_days
0,abadia de goias,GO,sobral,CE,1,43.410,24
1,afonso claudio,ES,belem,PA,1,29.620,20
2,afonso claudio,ES,franca,SP,1,17.190,11
3,afonso claudio,ES,macae,RJ,1,15.560,6
4,afonso claudio,ES,niteroi,RJ,1,17.430,7
...,...,...,...,...,...,...,...
35879,xanxere,SC,santanesia,RJ,1,17.950,20
35880,xanxere,SC,sao paulo,SP,2,25.075,9
35881,xanxere,SC,sete lagoas,MG,1,23.410,14
35882,xaxim,SC,cruz alta,RS,1,18.020,11


In [ ]:
data_sellers = data_orders_[['order_id', 'seller_id', 'seller_city', 'seller_state', 'order_status', 'is_late', 'review_score', 'approved_mins']]\
    .drop_duplicates()\
    .groupby (by = ['seller_id', 'seller_city', 'seller_state'])\
    .apply (lambda x: pd.Series ({
        'total_orders': x['order_id'].nunique(),
        'delivered_orders': x.query("order_status == 'delivered'")['order_id'].nunique(),
        'cancelled_orders': x.query("order_status == 'canceled'")['order_id'].nunique(),
        'late_delivery': x.query("is_late == True")['order_id'].nunique(),
        'avg_approved_mins': x['approved_mins'].mean(),
        'review_score': x.query("review_score != 0")['review_score'].mean(),
    }))\
    .assign(
        delivered_ratio = lambda x: round (x['delivered_orders']/x['total_orders']*100, 2),
        cancelled_ratio = lambda x: round (x['cancelled_orders']/x['total_orders']*100, 2),
    )\
    .reset_index()

data_sellers.head()

,seller_id,seller_city,seller_state,total_orders,delivered_orders,cancelled_orders,late_delivery,avg_approved_mins,review_score,delivered_ratio,cancelled_ratio
0,0015a82c2db000af6aaaf3ae2ecb0532,santo andre,SP,3.0,3.0,0.0,0.0,800.666667,3.666667,100.00,0.0
1,001cca7ae9ae17fb1caed9dfb1094831,cariacica,ES,200.0,195.0,0.0,13.0,591.290000,3.984772,97.50,0.0
2,001e6ad469a905060d959994f1b41e4f,sao goncalo,RJ,1.0,0.0,1.0,0.0,14.000000,1.000000,0.00,100.0
3,002100f778ceb8431b7a1020ff7ab48f,franca,SP,51.0,50.0,0.0,9.0,1376.313725,3.901961,98.04,0.0
4,003554e2dce176b5555353e4f3555ac8,goiania,GO,1.0,1.0,0.0,0.0,18.000000,5.000000,100.00,0.0


In [ ]:
data_items = df_order_items.merge (df_products[['product_id', 'product_category_name']], how = 'left', on = 'product_id')\
    .merge (df_product_category_name_translation, how = 'left', on = 'product_category_name')\
    .groupby (by = ['seller_id', 'product_id', 'product_category_name_english'])\
    .agg(
        product_count = pd.NamedAgg (column = 'product_id', aggfunc = 'count')
    )\
    .sort_values (by = 'product_count', ascending = False)\
    .reset_index()

data_items.head()

    # .groupby (by = 'seller_id')\
    # .apply (lambda x: pd.Series ({
    #     'products_sold': x[['product_id', 'product_count']].to_dict('records')
    # }))\
    # .reset_index()

,seller_id,product_id,product_category_name_english,product_count
0,955fee9216a65b617aa5c0531780ce60,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,527
1,1f50f920176fa81dab994f9023523100,422879e10f46682990de24d770e7f83d,garden_tools,484
2,4a3ca9315b744ce9f8e9374361493884,99a4788cb24856965c36a24e339b6058,bed_bath_table,482
3,1f50f920176fa81dab994f9023523100,389d119b48cf3043d311335e499d9c6b,garden_tools,392
4,1f50f920176fa81dab994f9023523100,368c6c730842d78016ad823897a372db,garden_tools,388


In [ ]:
def suggest (city: str, state: str, product_category, limit = 10, product_id = None):
    '''
    Function to suggest sellers based on customer's city and state and search on product's category
    '''

    # list all suggested sellers who have that product category on hand
    if product_id is not None:
        df_product = data_items[(data_items['product_id'] == product_id) & (data_items['product_category_name_english'] == product_category)]
    else:
        df_product = data_items[(data_items['product_category_name_english'] == product_category)]

    df_product = df_product.groupby (by = 'seller_id')\
        .apply (lambda x: pd.Series ({
            'products_sold': x[['product_id', 'product_count']].to_dict('records')
        }))

    # get other information about sellers' operation
    df = df_product.merge (data_sellers, how = 'left', on = 'seller_id')

    # get information about customers' with sellers' location
    df_location = data_shipping[(data_shipping['customer_city'] == city) & (data_shipping['customer_state'] == state)]
    df = df.merge(df_location, how  = 'inner', on = ['seller_city', 'seller_state'])\
        .sort_values (by = ['delivered_ratio', 'avg_delivery_days', 'avg_approved_mins'], ascending = [False, True, True])\
        .reset_index ()

    df = df[['seller_id', 'seller_city', 'seller_state', 'customer_city', 'customer_state', 'products_sold',
       'total_orders', 'delivered_orders', 'cancelled_orders', 'late_delivery',
       'avg_approved_mins', 'review_score', 'delivered_ratio',
       'cancelled_ratio', 'orders_num_same_route', 'avg_freight_value', 'avg_delivery_days']]

    return df

# For testing
suggest (product_category = 'garden_tools', city = 'sao goncalo', state = 'RJ')

,seller_id,seller_city,seller_state,customer_city,customer_state,products_sold,total_orders,delivered_orders,cancelled_orders,late_delivery,avg_approved_mins,review_score,delivered_ratio,cancelled_ratio,orders_num_same_route,avg_freight_value,avg_delivery_days
0,2aa3443d7bf9d9bb11133f420d75e083,rio de janeiro,RJ,sao goncalo,RJ,[{'product_id': '3630657a252ca88c694edfa53f2ad...,10.0,10.0,0.0,2.0,243.500000,3.00,100.0,0.0,12,9.973333,4.500000
1,a4bd6e9adf39b63f43dc545d3ca1f53d,rio de janeiro,RJ,sao goncalo,RJ,[{'product_id': 'e6baba6c7819d44817a76305e082d...,4.0,4.0,0.0,0.0,384.250000,4.50,100.0,0.0,12,9.973333,4.500000
2,7901646fdd36a55f564ffaf2dbccaaf7,rio de janeiro,RJ,sao goncalo,RJ,[{'product_id': '636598095d69a5718e67d2c9a3c7d...,22.0,22.0,0.0,3.0,607.045455,4.50,100.0,0.0,12,9.973333,4.500000
3,0a198e95d32b1be2da9424c962a6ebfa,contagem,MG,sao goncalo,RJ,[{'product_id': 'f0788219c3d63c6183bbca7699ca9...,1.0,1.0,0.0,0.0,27.000000,5.00,100.0,0.0,3,28.963333,4.666667
4,8a9260f2b0340411d6d2a56bcf4f7378,contagem,MG,sao goncalo,RJ,[{'product_id': 'b800d7bb8cd5a7093dd099a367d1d...,8.0,8.0,0.0,1.0,630.875000,4.75,100.0,0.0,3,28.963333,4.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,e628d4a53c109f09ca88098338b3a3f5,belo horizonte,MG,sao goncalo,RJ,[{'product_id': '57a2e16f66595e9242fd508a9f336...,1.0,0.0,1.0,0.0,7.000000,1.00,0.0,100.0,9,15.745556,14.777778
159,666658b8da8370f30e1f89893b1de5e6,sao paulo,SP,sao goncalo,RJ,[{'product_id': '33ac889bc3af4ddede9c14fc789a3...,1.0,0.0,1.0,0.0,970.000000,5.00,0.0,100.0,78,18.383671,15.594937
160,dff87e4de60c9736ce8df835951b09bc,sao paulo,SP,sao goncalo,RJ,[{'product_id': 'fc83db05de6120d00e02ea0fe6f91...,2.0,0.0,1.0,0.0,970.500000,2.50,0.0,50.0,78,18.383671,15.594937
161,ac1ed5fc15901fbc92920361eb4ab350,sao paulo,SP,sao goncalo,RJ,[{'product_id': '6548e4e34dccfc63407b2bddc88b1...,1.0,0.0,0.0,0.0,4131.000000,1.00,0.0,0.0,78,18.383671,15.594937
